# KYC — Extraction d'identité / Identity extraction pipeline

Local, offline pipeline for KYC folders:

1. **Unzip** `kyc_documents.zip` and keep only the 5 required PDFs per customer folder.
2. **Presence report** (`presence_report.csv` / `.xlsx`): which of the 5 documents exist per customer.
3. **Target customers** = folders where `JUSTIFICATIF IDENTITE.PDF` exists.
4. **Rasterise** every page of `JUSTIFICATIF IDENTITE.PDF` (multi-page scans supported).
5. **Geometric correction only** (orientation 0/90/180/270 + deskew). No content-inventing enhancement.
6. **Extract** with the locally loaded FP8 VLM using the strict anti-hallucination system prompt.
7. **Merge pages** field-by-field with confidence ranking and conflict detection (conflicts → `null`).
8. Write one schema-clean JSON per customer + an audit sidecar + a consolidated CSV.

**Design principle:** the pipeline never repairs, completes or normalises a value.
Anything not visually supported stays `null`. Conflicts between pages are resolved to `null`,
never to the "more plausible" value.

Models used (both local, no network):

- VLM: `/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main` (FP8)
- FP8 kernels: `/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8/v4`

## 0. Configuration

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- model paths
MODEL_PATH  = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"
KERNEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8/v4"

BACKEND          = "vllm"      # "vllm" (recommended for FP8) or "transformers"
TENSOR_PARALLEL  = 1           # set to the number of GPUs available
GPU_MEM_UTIL     = 0.90
MAX_MODEL_LEN    = 8192
MAX_NEW_TOKENS   = 2048
RUN_MODEL        = True        # False -> run steps 1-3 only (inventory / rendering), no GPU needed

# ------------------------------------------------------------------ io paths
ZIP_PATH   = Path("kyc_documents.zip")
WORK_DIR   = Path("kyc_work")
EXTRACT_DIR = WORK_DIR / "01_extracted"     # the 5 kept PDFs, per customer
PAGES_DIR   = WORK_DIR / "02_pages"         # rasterised + corrected page images
RAW_DIR     = WORK_DIR / "03_raw_model_out" # verbatim model output per page
OUT_DIR     = WORK_DIR / "04_results"       # final JSON per customer + audit
REPORT_DIR  = WORK_DIR / "05_reports"

for d in (EXTRACT_DIR, PAGES_DIR, RAW_DIR, OUT_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------- rasterisation
RENDER_DPI   = 300      # 300 dpi is the floor for small print / MRZ
MAX_SIDE_PX  = 3200     # downscale guard so a page fits the vision encoder
MIN_SIDE_PX  = 900      # below this the page is flagged as low resolution

# --------------------------------------------------------- documents of interest
# canonical name -> token set that must all be present in the normalised file stem.
# Note: "CARTON SIGNATUTE" is the spelling used by the business; the typo-free
# "CARTON SIGNATURE" is accepted as well.
REQUIRED_DOCS = {
    "JUSTIFICATIF_IDENTITE": [{"JUSTIFICATIF", "IDENTITE"}],
    "JUSTIFICATIF_DOMICILE": [{"JUSTIFICATIF", "DOMICILE"}],
    "CONVENTION_COMPTE":     [{"CONVENTION", "COMPTE"}],
    "FATCA":                 [{"FATCA"}],
    "CARTON_SIGNATURE":      [{"CARTON", "SIGNATUTE"}, {"CARTON", "SIGNATURE"}],
}
IDENTITY_DOC = "JUSTIFICATIF_IDENTITE"

print("work dir:", WORK_DIR.resolve())

In [ ]:
import os

# Force fully offline execution: nothing must be fetched from the Hub.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# Point the `kernels` runtime at the locally downloaded finegrained-fp8 kernel.
os.environ.setdefault("KERNELS_CACHE", KERNEL_PATH)
os.environ.setdefault("HF_KERNELS_CACHE", KERNEL_PATH)
os.environ.setdefault("DISABLE_KERNEL_MAPPING", "0")

# Deterministic decoding helpers
os.environ.setdefault("PYTHONHASHSEED", "0")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import sys, json, re, zipfile, base64, hashlib, unicodedata, traceback, io, math
from datetime import datetime, timezone

import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

try:
    import cv2
    HAVE_CV2 = True
except Exception:
    HAVE_CV2 = False

try:
    import pandas as pd
    HAVE_PANDAS = True
except Exception:
    HAVE_PANDAS = False

print("python  :", sys.version.split()[0])
print("opencv  :", HAVE_CV2)
print("pandas  :", HAVE_PANDAS)
print("kernels :", KERNEL_PATH)

## 1. Unzip, keep only the 5 required PDFs, and build the presence report

Folder names are customer identifiers. File names are matched on a **normalised** form
(uppercase, accents stripped, punctuation/underscores collapsed to spaces) so that
`Justificatif_Identité.pdf`, `JUSTIFICATIF IDENTITE.PDF` and `justificatif-identite.pdf`
all match the same canonical document.

In [ ]:
def normalise_name(text: str) -> str:
    """Uppercase, strip accents, collapse anything non-alphanumeric to single spaces."""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def classify_document(filename: str):
    """Return the canonical document key for a file name, or None if not of interest."""
    stem = Path(filename).stem
    tokens = set(normalise_name(stem).split())
    for canonical, variants in REQUIRED_DOCS.items():
        for required in variants:
            if required.issubset(tokens):
                return canonical
    return None


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_member(name: str) -> bool:
    """Reject absolute paths and traversal (zip-slip)."""
    p = Path(name)
    return not p.is_absolute() and ".." not in p.parts

In [ ]:
def extract_kyc_zip(zip_path: Path, dest: Path):
    """
    Extract only the 5 documents of interest.
    Returns (inventory, ignored) where inventory maps customer_id -> {doc_key: [paths]}.
    """
    inventory, ignored = {}, []
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if info.is_dir() or not safe_member(info.filename):
                continue
            parts = Path(info.filename).parts
            if len(parts) < 2:
                ignored.append(info.filename)          # loose file at zip root
                continue
            customer_id = parts[-2]                     # folder directly containing the file
            fname = parts[-1]
            if Path(fname).suffix.lower() != ".pdf":
                ignored.append(info.filename)
                continue
            doc_key = classify_document(fname)
            if doc_key is None:
                ignored.append(info.filename)
                continue

            target_dir = dest / customer_id
            target_dir.mkdir(parents=True, exist_ok=True)
            target = target_dir / f"{doc_key}.pdf"
            n = 1
            while target.exists():                      # duplicates are kept, never overwritten
                target = target_dir / f"{doc_key}__dup{n}.pdf"
                n += 1
            with zf.open(info) as src, open(target, "wb") as out:
                out.write(src.read())
            inventory.setdefault(customer_id, {}).setdefault(doc_key, []).append(target)

        # customers whose folder exists in the zip but held none of the 5 documents
        for info in zf.infolist():
            if info.is_dir() and safe_member(info.filename):
                cid = Path(info.filename).name
                if cid:
                    inventory.setdefault(cid, {})
    return inventory, ignored


if ZIP_PATH.exists():
    INVENTORY, IGNORED = extract_kyc_zip(ZIP_PATH, EXTRACT_DIR)
else:
    # already-unzipped fallback: EXTRACT_DIR/<customer>/<doc>.pdf
    INVENTORY, IGNORED = {}, []
    for folder in sorted(p for p in EXTRACT_DIR.iterdir() if p.is_dir()):
        docs = {}
        for pdf in folder.glob("*.pdf"):
            key = classify_document(pdf.name)
            if key:
                docs.setdefault(key, []).append(pdf)
        INVENTORY[folder.name] = docs
    print(f"WARNING: {ZIP_PATH} not found — reusing already extracted folders.")

print(f"customers: {len(INVENTORY)}   files ignored (not in scope): {len(IGNORED)}")

In [ ]:
rows = []
for customer_id in sorted(INVENTORY):
    docs = INVENTORY[customer_id]
    row = {"customer_id": customer_id}
    for key in REQUIRED_DOCS:
        paths = docs.get(key, [])
        row[key] = "PRESENT" if paths else "MISSING"
        row[f"{key}__n_files"] = len(paths)
        row[f"{key}__path"] = str(paths[0]) if paths else ""
    row["n_present"] = sum(1 for k in REQUIRED_DOCS if docs.get(k))
    row["n_missing"] = len(REQUIRED_DOCS) - row["n_present"]
    row["complete_folder"] = row["n_missing"] == 0
    rows.append(row)

presence_csv = REPORT_DIR / "presence_report.csv"

if HAVE_PANDAS:
    presence_df = pd.DataFrame(rows)
    presence_df.to_csv(presence_csv, index=False, encoding="utf-8-sig")
    try:
        presence_df.to_excel(REPORT_DIR / "presence_report.xlsx", index=False)
    except Exception as exc:
        print("xlsx export skipped:", exc)
    display(presence_df[["customer_id"] + list(REQUIRED_DOCS) + ["n_missing"]].head(20))
else:
    import csv
    with open(presence_csv, "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    presence_df = None

with open(REPORT_DIR / "presence_report.json", "w", encoding="utf-8") as fh:
    json.dump(rows, fh, ensure_ascii=False, indent=2)

print("presence report ->", presence_csv)

In [ ]:
TARGET_CUSTOMERS = [c for c in sorted(INVENTORY) if INVENTORY[c].get(IDENTITY_DOC)]

with open(REPORT_DIR / "target_customers.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join(TARGET_CUSTOMERS))

print(f"target customers (JUSTIFICATIF IDENTITE present): {len(TARGET_CUSTOMERS)} / {len(INVENTORY)}")
print(TARGET_CUSTOMERS[:10])

## 2. Rasterise the identity document

Identity proofs are scans, so the text layer (when any) is unreliable — every page is
rendered as an image at 300 dpi and read visually by the VLM.
`pypdfium2` is preferred (pure wheel), with `PyMuPDF` and `pdftoppm` as fallbacks.

In [ ]:
_RENDERER = None
try:
    import pypdfium2 as pdfium
    _RENDERER = "pypdfium2"
except Exception:
    try:
        import fitz  # PyMuPDF
        _RENDERER = "pymupdf"
    except Exception:
        import shutil
        if shutil.which("pdftoppm"):
            _RENDERER = "pdftoppm"
print("renderer:", _RENDERER)


def render_pdf_pages(pdf_path: Path, out_dir: Path, dpi: int = RENDER_DPI):
    """Rasterise every page to RGB PNG. Returns the list of page image paths."""
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []

    if _RENDERER == "pypdfium2":
        doc = pdfium.PdfDocument(str(pdf_path))
        try:
            for i in range(len(doc)):
                img = doc[i].render(scale=dpi / 72).to_pil().convert("RGB")
                p = out_dir / f"page_{i+1:03d}.png"
                img.save(p); paths.append(p)
        finally:
            doc.close()

    elif _RENDERER == "pymupdf":
        doc = fitz.open(str(pdf_path))
        try:
            for i, page in enumerate(doc):
                pix = page.get_pixmap(dpi=dpi)
                img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
                p = out_dir / f"page_{i+1:03d}.png"
                img.save(p); paths.append(p)
        finally:
            doc.close()

    elif _RENDERER == "pdftoppm":
        import subprocess
        subprocess.run(["pdftoppm", "-png", "-r", str(dpi), str(pdf_path),
                        str(out_dir / "page")], check=True)
        paths = sorted(out_dir.glob("page-*.png"))
    else:
        raise RuntimeError("No PDF renderer available (pypdfium2 / PyMuPDF / poppler).")

    return paths

## 3. Geometric correction and image-quality metrics

Only **geometry** is corrected (90° orientation + small skew). Contrast/denoise are *not*
applied to the image sent to the model: a pixel that is unreadable must stay unreadable so
the model reports `unreadable` instead of hallucinating a "cleaned up" character.
Quality metrics are measured and carried into the `image_quality` block of the output.

In [ ]:
def image_quality_metrics(img: Image.Image) -> dict:
    """Objective, model-independent quality indicators for a page."""
    g = np.asarray(img.convert("L"), dtype=np.float32)
    h, w = g.shape

    if HAVE_CV2:
        blur = float(cv2.Laplacian(g, cv2.CV_32F).var())
    else:
        gx = np.diff(g, axis=1); gy = np.diff(g, axis=0)
        blur = float(gx.var() + gy.var())

    p2, p98 = np.percentile(g, [2, 98])
    contrast = float(p98 - p2)
    dark = float((g < 60).mean())

    issues = []
    if min(h, w) < MIN_SIDE_PX:   issues.append("low_resolution")
    if blur < 60:                 issues.append("blurred")
    if contrast < 45:             issues.append("low_contrast")
    if dark > 0.55:               issues.append("shadow_or_dark_scan")

    if not issues:                        overall = "good"
    elif len(issues) == 1:                overall = "degraded"
    else:                                 overall = "poor"

    return {"width": int(w), "height": int(h), "blur_score": round(blur, 2),
            "contrast_score": round(contrast, 2), "dark_ratio": round(dark, 3),
            "issues": issues, "overall": overall}


def deskew(img: Image.Image, max_angle: float = 12.0):
    """Correct small rotations (< max_angle degrees). Returns (image, angle_applied)."""
    if not HAVE_CV2:
        return img, 0.0
    g = np.asarray(img.convert("L"))
    thr = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    thr = cv2.morphologyEx(thr, cv2.MORPH_CLOSE, np.ones((3, 15), np.uint8))
    coords = cv2.findNonZero(thr)
    if coords is None or len(coords) < 200:
        return img, 0.0
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:  angle += 90
    if angle > 45:   angle -= 90
    if abs(angle) < 0.3 or abs(angle) > max_angle:
        return img, 0.0
    arr = np.asarray(img.convert("RGB"))
    h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rot = cv2.warpAffine(arr, M, (w, h), flags=cv2.INTER_CUBIC,
                         borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rot), float(angle)


def cap_size(img: Image.Image, max_side: int = MAX_SIDE_PX) -> Image.Image:
    if max(img.size) <= max_side:
        return img
    ratio = max_side / max(img.size)
    return img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)

In [ ]:
ORIENTATION_PROMPT = (
    "Look at this scanned page. Determine only how the page is rotated relative to "
    "upright, readable text. Answer with strict JSON and nothing else: "
    '{"rotate_clockwise_degrees": 0} or 90 or 180 or 270, where the value is the '
    "rotation to apply clockwise to make the text upright."
)

def detect_orientation_tesseract(img: Image.Image):
    """0/90/180/270 clockwise rotation to apply, or None if unavailable/uncertain."""
    try:
        import pytesseract
        osd = pytesseract.image_to_osd(img, output_type=pytesseract.Output.DICT)
        if float(osd.get("orientation_conf", 0)) < 1.0:
            return None
        return int(osd.get("rotate", 0)) % 360
    except Exception:
        return None


def detect_orientation_vlm(img: Image.Image, engine):
    """Fallback: ask the loaded VLM itself. Cheap, single short answer."""
    try:
        txt = engine.generate(cap_size(img, 1024), ORIENTATION_PROMPT,
                              system="You answer with strict JSON only.", max_tokens=32)
        m = re.search(r"(\d{1,3})", txt)
        if m:
            val = int(m.group(1))
            return val if val in (0, 90, 180, 270) else 0
    except Exception:
        pass
    return 0


def prepare_page(img: Image.Image, engine=None):
    """Orientation -> deskew -> size cap. Returns (image, transform log, quality metrics)."""
    log = {}
    rot = detect_orientation_tesseract(img)
    log["orientation_method"] = "tesseract_osd" if rot is not None else "none"
    if rot is None and engine is not None:
        rot = detect_orientation_vlm(img, engine)
        log["orientation_method"] = "vlm_probe"
    rot = rot or 0
    if rot:
        img = img.rotate(-rot, expand=True)   # PIL rotates counter-clockwise
    log["rotation_applied_deg"] = rot

    img, angle = deskew(img)
    log["deskew_applied_deg"] = round(angle, 2)

    img = cap_size(img)
    return img, log, image_quality_metrics(img)

## 4. Prompts

The system prompt is the forensic-transcription contract. It is sent **verbatim** with every
page. Nothing in the pipeline is allowed to relax it.

In [ ]:
SYSTEM_PROMPT = """You are a high-precision OCR and document extraction engine specialized in identity documents such as passports, national identity cards, residence permits, and driver's licenses.

Your primary objective is to extract ONLY information that is visually supported by the supplied document image.

CRITICAL ANTI-HALLUCINATION RULES

NEVER guess a character, digit, word, name, date, or document number.
NEVER infer missing characters from context.
NEVER complete partially visible text.
NEVER correct spelling, transliteration, formatting, or apparent OCR errors.
NEVER use external knowledge to reconstruct unreadable text.
NEVER assume what a field "should" contain based on the document type.
If a character cannot be distinguished reliably from the image, mark that character as uncertain.
If a complete field cannot be read reliably, return null.
A partially unreadable value is preferable to an invented complete value.
Do NOT manufacture a value merely because the requested JSON field requires a value.
Do NOT answer with explanations, guesses, or conversational text.
The image is the ONLY authoritative source of the extracted value.

VISUAL EVIDENCE RULE

For every extracted value, ask internally:
"Can I directly see every character of this value in the image?"
If the answer is NO:
- Do not guess.
- Do not infer.
- Do not reconstruct.
- Return the value as null or mark the uncertain character(s).

For example, if the image appears to contain:
A12?45B7
and the fourth character cannot be reliably distinguished, DO NOT output:
A12345B7
Instead output:
{"value": null, "status": "uncertain", "uncertain_positions": [4]}

CHARACTER-LEVEL TRANSCRIPTION

Preserve exactly what is visually present.
Do not:
- fix spelling
- normalize names
- translate names
- expand abbreviations
- change Arabic names into a preferred Latin spelling
- replace visually similar characters unless the image clearly establishes the character

Pay particular attention to visually similar characters:
0 / O
1 / I / L
2 / Z
5 / S
6 / G
8 / B
C / G
U / V
D / O
M / N

If the image does not allow a reliable distinction, mark the character as uncertain.

DOCUMENT FIELDS

Extract only the fields that are requested.
Typical fields may include: surname, given_names, date_of_birth, place_of_birth, nationality, sex, document_number, issue_date, expiry_date, issuing_authority, personal_number, MRZ.
Do not invent fields that are not visible.

MRZ

If a passport contains a Machine Readable Zone:
- Transcribe the MRZ exactly as visible.
- Do not reconstruct missing characters.
- Preserve < characters.
- Do not "repair" the MRZ simply because the expected ICAO format suggests another character.
- If a character is unreadable, mark it as uncertain.
- If the MRZ is too degraded to reliably transcribe, return null.
- If MRZ validation is performed, validation may identify an inconsistency but must NEVER be used to invent the missing character.

IMAGE QUALITY

Before extraction, evaluate: resolution, blur, compression artifacts, contrast, skew, cropping, shadows, missing portions, character visibility.
If image quality prevents reliable extraction, report that instead of guessing.
Do not assume that image enhancement makes an unreadable character readable.

FIELD-LEVEL CONFIDENCE

For each field, assign one of:
high: every character is clearly visible
medium: most characters are visible but one or more are somewhat ambiguous
low: significant ambiguity exists
unreadable: the value cannot be reliably extracted

Confidence refers to VISUAL EVIDENCE, not how plausible the resulting value appears.
A plausible value with weak visual evidence must NOT receive high confidence.

OUTPUT REQUIREMENT

Return ONLY valid JSON.
Never return Markdown. Never return explanations. Never return commentary before or after the JSON.

Use this structure:

{
"document_type": {"value": null, "confidence": "unreadable"},
"surname": {"value": null, "confidence": "unreadable"},
"given_names": {"value": null, "confidence": "unreadable"},
"date_of_birth": {"value": null, "confidence": "unreadable"},
"place_of_birth": {"value": null, "confidence": "unreadable"},
"nationality": {"value": null, "confidence": "unreadable"},
"sex": {"value": null, "confidence": "unreadable"},
"document_number": {"value": null, "confidence": "unreadable"},
"issue_date": {"value": null, "confidence": "unreadable"},
"expiry_date": {"value": null, "confidence": "unreadable"},
"issuing_authority": {"value": null, "confidence": "unreadable"},
"mrz": {"value": null, "confidence": "unreadable"},
"image_quality": {"overall": "unreadable", "reason": null}
}

MOST IMPORTANT PRINCIPLE

When forced to choose between:
A. returning an incomplete/uncertain result
and
B. returning a plausible but unsupported result
ALWAYS choose A.

False information is worse than missing information.
The correct behavior for an unreadable document is: null, not a guess.
The model must behave like a forensic transcription system, not like a conversational assistant."""


USER_PROMPT = """Extract the identity information from this document according to the OCR rules in your system instructions.

Read the image directly.
Do not infer or reconstruct anything that is not clearly visible.

For every requested field:
- Locate the corresponding field in the document.
- Read the characters directly from the pixels.
- Verify every character visually.
- If one or more characters cannot be reliably distinguished, do not guess them.
- If the field cannot be reliably read, return null.
- Assign confidence based strictly on visual evidence.

Pay particular attention to:
- document number
- dates
- names
- visually similar letters and digits
- MRZ characters
- characters damaged by blur, compression, low contrast, or scanning artifacts.

Return ONLY the JSON object defined in the system instructions."""

print(len(SYSTEM_PROMPT), "chars system prompt |", len(USER_PROMPT), "chars user prompt")

## 5. Load the local FP8 model

`vllm` is used by default with `quantization="fp8"` pointing at the local snapshot;
the `transformers` backend is provided as a fallback and wires the finegrained-fp8 kernel
through the `kernels` runtime. Decoding is greedy (`temperature=0`) — sampling is a source
of invented characters and is never enabled here.

In [ ]:
def pil_to_data_uri(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="PNG", optimize=False)  # lossless: no JPEG artifacts
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


class VLMEngine:
    """Thin wrapper over the locally loaded FP8 vision-language model."""

    def __init__(self, backend=BACKEND, model_path=MODEL_PATH):
        self.backend = backend
        self.model_path = model_path
        if backend == "vllm":
            from vllm import LLM, SamplingParams
            self._SamplingParams = SamplingParams
            self.llm = LLM(
                model=model_path,
                tokenizer=model_path,
                quantization="fp8",
                trust_remote_code=True,
                dtype="auto",
                tensor_parallel_size=TENSOR_PARALLEL,
                gpu_memory_utilization=GPU_MEM_UTIL,
                max_model_len=MAX_MODEL_LEN,
                limit_mm_per_prompt={"image": 1},
                enforce_eager=False,
            )
        elif backend == "transformers":
            import torch
            from transformers import AutoProcessor, AutoModelForImageTextToText
            self.torch = torch
            self.processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
            kwargs = dict(torch_dtype="auto", device_map="auto", trust_remote_code=True)
            try:                                  # enable the finegrained-fp8 kernel
                self.model = AutoModelForImageTextToText.from_pretrained(
                    model_path, use_kernels=True, **kwargs)
            except TypeError:
                self.model = AutoModelForImageTextToText.from_pretrained(model_path, **kwargs)
            self.model.eval()
        else:
            raise ValueError(f"unknown backend {backend}")

    def generate(self, image: Image.Image, prompt: str, system: str = SYSTEM_PROMPT,
                 max_tokens: int = MAX_NEW_TOKENS) -> str:
        if self.backend == "vllm":
            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": pil_to_data_uri(image)}},
                    {"type": "text", "text": prompt},
                ]},
            ]
            sp = self._SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_tokens, seed=0)
            out = self.llm.chat(messages, sampling_params=sp)
            return out[0].outputs[0].text

        messages = [
            {"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": [
                {"type": "image", "image": image.convert("RGB")},
                {"type": "text", "text": prompt},
            ]},
        ]
        inputs = self.processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt").to(self.model.device)
        with self.torch.inference_mode():
            ids = self.model.generate(**inputs, do_sample=False, temperature=None, top_p=None,
                                      max_new_tokens=max_tokens)
        new = ids[0][inputs["input_ids"].shape[-1]:]
        return self.processor.decode(new, skip_special_tokens=True)


ENGINE = VLMEngine() if RUN_MODEL else None
print("engine:", "loaded" if ENGINE else "disabled (RUN_MODEL=False)")

## 6. Strict output parsing

The model must return JSON. The parser **does not repair values** — it only locates the JSON
object and validates the schema. Anything malformed becomes an all-`null` result flagged as
`model_output_not_valid_json`, which is the safe failure mode.

In [ ]:
FIELDS = ["document_type", "surname", "given_names", "date_of_birth", "place_of_birth",
          "nationality", "sex", "document_number", "issue_date", "expiry_date",
          "issuing_authority", "mrz"]
CONFIDENCES = ("high", "medium", "low", "unreadable")
CONF_RANK = {"high": 3, "medium": 2, "low": 1, "unreadable": 0}


def empty_result(reason=None):
    r = {f: {"value": None, "confidence": "unreadable"} for f in FIELDS}
    r["image_quality"] = {"overall": "unreadable", "reason": reason}
    return r


def find_json_object(text: str):
    """Return the first balanced {...} block, ignoring braces inside strings."""
    start = text.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:            esc = False
            elif ch == "\\":   esc = True
            elif ch == '"':    in_str = False
            continue
        if ch == '"':   in_str = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def parse_model_json(text: str):
    """Validate the model output against the schema. Never rewrites a value."""
    blob = find_json_object(text or "")
    if blob is None:
        return empty_result("model_output_not_valid_json"), ["no_json_found"]
    try:
        data = json.loads(blob)
    except Exception:
        return empty_result("model_output_not_valid_json"), ["json_decode_error"]
    if not isinstance(data, dict):
        return empty_result("model_output_not_valid_json"), ["json_not_object"]

    flags, out = [], {}
    for f in FIELDS:
        item = data.get(f)
        if isinstance(item, dict):
            value = item.get("value", None)
            conf = item.get("confidence", None)
            extra = {k: v for k, v in item.items()
                     if k in ("status", "uncertain_positions") and v not in (None, [])}
        elif isinstance(item, (str, int, float)) or item is None:
            value, conf, extra = item, None, {}     # tolerate a bare value, keep it verbatim
            if item is not None:
                flags.append(f"{f}:missing_confidence_wrapper")
        else:
            value, conf, extra = None, "unreadable", {}
            flags.append(f"{f}:unexpected_type")

        if isinstance(value, (int, float)):
            value = str(value)
        if isinstance(value, str):
            value = value.strip()
            if value == "" or value.lower() in ("null", "none", "n/a", "-"):
                value = None
        elif value is not None:
            value = None
            flags.append(f"{f}:non_scalar_value")

        if conf not in CONFIDENCES:
            conf = "unreadable" if value is None else "low"   # never upgrade
            flags.append(f"{f}:invalid_confidence")
        if value is None and conf != "unreadable":
            conf = "unreadable"
        out[f] = {"value": value, "confidence": conf, **extra}

    iq = data.get("image_quality") if isinstance(data.get("image_quality"), dict) else {}
    out["image_quality"] = {"overall": iq.get("overall") or "unreadable",
                            "reason": iq.get("reason")}

    unknown = [k for k in data if k not in FIELDS + ["image_quality"]]
    if unknown:
        flags.append("unknown_fields:" + ",".join(map(str, unknown[:5])))
    return out, flags

## 7. Per-page extraction and multi-page merge

A `JUSTIFICATIF IDENTITE.PDF` is typically the recto/verso (or several documents) spread over
pages. Each page is read independently, then merged:

- the highest-confidence non-null candidate wins;
- **equal-rank candidates that disagree resolve to `null`** with the conflict recorded;
- the page that produced each value is stored in the audit file.

In [ ]:
def extract_page(engine, page_img: Image.Image, raw_path: Path):
    raw = engine.generate(page_img, USER_PROMPT)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(raw, encoding="utf-8")     # verbatim model output kept for audit
    parsed, flags = parse_model_json(raw)
    return parsed, flags


def mrz_check_report(mrz_value):
    """ICAO check digits, for AUDIT ONLY. Never used to alter or complete a value."""
    if not mrz_value or not isinstance(mrz_value, str):
        return None
    lines = [l.strip().upper() for l in mrz_value.splitlines() if l.strip()]
    if len(lines) != 2 or len({len(l) for l in lines}) != 1 or len(lines[0]) not in (44, 36):
        return {"validated": False, "reason": "unexpected_mrz_layout"}
    weights, vals = [7, 3, 1], {}
    for i, c in enumerate("0123456789"): vals[c] = i
    for i, c in enumerate("ABCDEFGHIJKLMNOPQRSTUVWXYZ"): vals[c] = 10 + i
    vals["<"] = 0

    def cd(s):
        if any(c not in vals for c in s):
            return None
        return sum(vals[c] * weights[i % 3] for i, c in enumerate(s)) % 10

    l2 = lines[1]
    checks = {}
    if len(l2) == 44:  # TD3 / passport
        spec = {"document_number": (0, 9, 9), "date_of_birth": (13, 19, 19),
                "expiry_date": (21, 27, 27)}
        for name, (a, b, d) in spec.items():
            calc, seen = cd(l2[a:b]), l2[d]
            checks[name] = None if calc is None or not seen.isdigit() else (str(calc) == seen)
    return {"validated": True, "check_digits_match": checks,
            "note": "informational only; discrepancies must never be auto-corrected"}


def merge_pages(page_results):
    """Field-wise merge across pages. Conflicts -> null."""
    final, audit = {}, {}
    for f in FIELDS:
        cands = []
        for idx, res in enumerate(page_results, start=1):
            item = res.get(f, {})
            if item.get("value") is not None:
                cands.append({"page": idx, "value": item["value"],
                              "confidence": item.get("confidence", "low"),
                              "status": item.get("status"),
                              "uncertain_positions": item.get("uncertain_positions")})
        if not cands:
            final[f] = {"value": None, "confidence": "unreadable"}
            audit[f] = {"candidates": [], "decision": "no_visual_evidence_on_any_page"}
            continue

        best_rank = max(CONF_RANK.get(c["confidence"], 0) for c in cands)
        top = [c for c in cands if CONF_RANK.get(c["confidence"], 0) == best_rank]
        distinct = {c["value"] for c in top}

        if len(distinct) == 1:
            chosen = top[0]
            item = {"value": chosen["value"], "confidence": chosen["confidence"]}
            if chosen.get("status"):               item["status"] = chosen["status"]
            if chosen.get("uncertain_positions"):  item["uncertain_positions"] = chosen["uncertain_positions"]
            final[f] = item
            audit[f] = {"candidates": cands, "decision": f"page_{chosen['page']}"}
        else:
            # two pages disagree at the same confidence: refuse to arbitrate
            final[f] = {"value": None, "confidence": "unreadable"}
            audit[f] = {"candidates": cands,
                        "decision": "conflict_between_pages_resolved_to_null"}
    return final, audit

## 8. Run the pipeline over the target customers

In [ ]:
def worst_quality(page_metrics):
    order = {"good": 0, "degraded": 1, "poor": 2, "unreadable": 3}
    worst, issues = "good", []
    for m in page_metrics:
        issues += m["issues"]
        if order.get(m["overall"], 0) > order.get(worst, 0):
            worst = m["overall"]
    return worst, sorted(set(issues))


def process_customer(customer_id: str, engine):
    pdf_path = INVENTORY[customer_id][IDENTITY_DOC][0]
    pages_dir = PAGES_DIR / customer_id
    page_files = render_pdf_pages(pdf_path, pages_dir)

    page_results, page_audit, metrics, flags = [], [], [], []
    for i, pf in enumerate(page_files, start=1):
        img = Image.open(pf).convert("RGB")
        img, transform, quality = prepare_page(img, engine)
        img.save(pages_dir / f"corrected_{i:03d}.png")
        metrics.append(quality)

        if engine is None:
            res, page_flags = empty_result("model_disabled"), ["model_disabled"]
        else:
            try:
                res, page_flags = extract_page(engine, img, RAW_DIR / customer_id / f"page_{i:03d}.txt")
            except Exception as exc:
                res, page_flags = empty_result("inference_error"), [f"inference_error:{exc}"]
        page_results.append(res)
        page_audit.append({"page": i, "source": str(pf), "transform": transform,
                           "quality_metrics": quality, "flags": page_flags,
                           "result": res})
        flags += [f"p{i}:{fl}" for fl in page_flags]

    final, field_audit = merge_pages(page_results)

    overall, issues = worst_quality(metrics)
    if all(final[f]["value"] is None for f in FIELDS):
        overall = "unreadable"
    final["image_quality"] = {"overall": overall,
                              "reason": ", ".join(issues) if issues else None}

    audit = {"customer_id": customer_id,
             "source_pdf": str(pdf_path),
             "sha256": sha256_of(pdf_path),
             "n_pages": len(page_files),
             "processed_at_utc": datetime.now(timezone.utc).isoformat(),
             "model": {"path": MODEL_PATH, "backend": BACKEND,
                       "quantization": "fp8", "kernel": KERNEL_PATH,
                       "decoding": "greedy/temperature=0"},
             "renderer": _RENDERER, "render_dpi": RENDER_DPI,
             "pages": page_audit, "field_provenance": field_audit,
             "mrz_check": mrz_check_report(final["mrz"]["value"]),
             "flags": flags}
    return final, audit


RESULTS, AUDITS, FAILURES = {}, {}, {}

for n, cid in enumerate(TARGET_CUSTOMERS, start=1):
    try:
        final, audit = process_customer(cid, ENGINE)
        RESULTS[cid], AUDITS[cid] = final, audit
        (OUT_DIR / f"{cid}.json").write_text(
            json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")
        (OUT_DIR / f"{cid}.audit.json").write_text(
            json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
        status = final["image_quality"]["overall"]
    except Exception as exc:
        FAILURES[cid] = traceback.format_exc()
        RESULTS[cid] = empty_result("processing_error")
        (OUT_DIR / f"{cid}.json").write_text(
            json.dumps(RESULTS[cid], ensure_ascii=False, indent=2), encoding="utf-8")
        status = "ERROR"
    print(f"[{n}/{len(TARGET_CUSTOMERS)}] {cid}: {status}")

print("done. failures:", len(FAILURES))

## 9. Consolidated outputs and QA report

In [ ]:
flat = []
for cid, res in RESULTS.items():
    row = {"customer_id": cid}
    for f in FIELDS:
        row[f] = res[f]["value"]
        row[f + "__confidence"] = res[f]["confidence"]
    row["image_quality"] = res["image_quality"]["overall"]
    row["image_quality_reason"] = res["image_quality"]["reason"]
    row["n_fields_extracted"] = sum(1 for f in FIELDS if res[f]["value"] is not None)
    row["n_fields_high_conf"] = sum(1 for f in FIELDS if res[f]["confidence"] == "high")
    row["needs_manual_review"] = (
        row["n_fields_extracted"] < 4
        or any(res[f]["confidence"] in ("low", "unreadable")
               for f in ("surname", "given_names", "document_number", "date_of_birth"))
    )
    flat.append(row)

with open(REPORT_DIR / "identity_extraction.jsonl", "w", encoding="utf-8") as fh:
    for cid, res in RESULTS.items():
        fh.write(json.dumps({"customer_id": cid, "extraction": res}, ensure_ascii=False) + "\n")

csv_path = REPORT_DIR / "identity_extraction.csv"
if HAVE_PANDAS and flat:
    df = pd.DataFrame(flat)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    display(df.head(20))
    print("\nmanual review queue:", int(df["needs_manual_review"].sum()), "/", len(df))
elif flat:
    import csv as _csv
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as fh:
        w = _csv.DictWriter(fh, fieldnames=list(flat[0].keys()))
        w.writeheader(); w.writerows(flat)

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "zip": str(ZIP_PATH),
    "customers_total": len(INVENTORY),
    "customers_with_identity_doc": len(TARGET_CUSTOMERS),
    "customers_processed": len(RESULTS),
    "processing_failures": list(FAILURES),
    "documents_missing_by_type": {
        k: sum(1 for c in INVENTORY if not INVENTORY[c].get(k)) for k in REQUIRED_DOCS},
    "field_fill_rate": {
        f: round(sum(1 for r in RESULTS.values() if r[f]["value"] is not None) / max(len(RESULTS), 1), 3)
        for f in FIELDS},
    "confidence_distribution": {
        f: {c: sum(1 for r in RESULTS.values() if r[f]["confidence"] == c) for c in CONFIDENCES}
        for f in FIELDS},
    "outputs": {"presence_report": str(presence_csv),
                "per_customer_json": str(OUT_DIR),
                "consolidated_csv": str(csv_path)},
}
(REPORT_DIR / "run_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2)[:2500])

In [ ]:
# Single-document inspection helper: print the exact JSON returned for one customer.
def show(customer_id):
    print(json.dumps(RESULTS[customer_id], ensure_ascii=False, indent=2))

if RESULTS:
    show(sorted(RESULTS)[0])

## Notes / operating guidance

- **`RUN_MODEL = False`** lets you validate steps 1–3 (unzip, presence report, rendering,
  orientation, quality metrics) on a CPU-only node before booking a GPU.
- **Backends.** `vllm` with `quantization="fp8"` is the fast path. If the checkpoint carries
  its own FP8 quant config, vLLM picks it up and the explicit `quantization` argument can be
  dropped. The `transformers` backend routes through the local `finegrained-fp8` kernel via
  `KERNELS_CACHE`/`use_kernels=True`.
- **Multilingual documents** need no special handling: the pages go to the VLM as pixels and
  the prompt forbids translation or transliteration, so Arabic/Latin mixed cards are
  transcribed as-seen.
- **What is deliberately *not* done:** no denoising, no binarisation, no super-resolution, no
  regex normalisation of dates or document numbers, no MRZ repair. MRZ check digits are
  computed for the audit file only and never write back into a value.
- **Conflicts** between recto and verso resolve to `null` with both candidates preserved in
  `<customer>.audit.json` — the reviewer arbitrates, not the pipeline.
- **Review queue:** `needs_manual_review` in `identity_extraction.csv` flags folders where the
  four identity-critical fields are not solidly readable.
- **Throughput:** for large batches, collect all page messages and call `llm.chat` once with a
  list of conversations — vLLM will batch them; keep `temperature=0` and `seed=0`.